### Hymenoptera Classification ###

In [ ]:
# Import libraries
import os

import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.data.experimental import cardinality
from tensorflow.keras import Input, Model, callbacks, layers, optimizers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.utils import image_dataset_from_directory

In [ ]:
TRAIN_PATH = "../data/train"
VAL_PATH = "../data/val"
RESULT_DIR = "../result"
MODEL_DIR = f"{RESULT_DIR}/model"
BEST_MODEL_PATH = f"{MODEL_DIR}/best_model.keras"

BATCH_SIZE = 16
IMG_SIZE = (160, 160)
RANDOM_STATE = 42
os.makedirs(MODEL_DIR, exist_ok=True)

In [ ]:
# Load dataset
train_dataset = image_dataset_from_directory(
    TRAIN_PATH,
    shuffle=True,
    seed=RANDOM_STATE,
    batch_size=BATCH_SIZE,
    image_size=IMG_SIZE,
)
val_dataset = image_dataset_from_directory(
    VAL_PATH, shuffle=False, batch_size=BATCH_SIZE, image_size=IMG_SIZE
)

In [ ]:
class_names = train_dataset.class_names
num_classes = len(train_dataset.class_names)

images_per_class = 8  # Number of images per class

total_images = num_classes * images_per_class

# Dictionary to store images for each class
images_by_class = {i: [] for i in range(num_classes)}

# Gather 8 images from each class
for image, label in train_dataset:
    label = label.numpy()
    for i in range(len(label)):
        class_idx = label[i]
        if len(images_by_class[class_idx]) < images_per_class:
            images_by_class[class_idx].append(image[i])

    # Stop when we have enough images
    if all(len(images) == images_per_class for images in images_by_class.values()):
        break

# Plot
plt.figure(figsize=(8, 8))

all_samples = [(c, img) for c, imgs in images_by_class.items() for img in imgs]
for count, (class_idx, img) in enumerate(all_samples):
    ax = plt.subplot(4, 4, count + 1)
    plt.imshow(img.numpy().astype("uint8"))
    plt.title(class_names[class_idx])
    plt.axis("off")

plt.savefig(f"{RESULT_DIR}/sample_images.png", bbox_inches="tight")
plt.show()

In [ ]:
# Split validation dataset to validation and testing dataset
# Shuffle once (reshuffle_each_iteration=False) so val and test never overlap between epochs
val_batches = int(cardinality(val_dataset))
test_batches = val_batches // 7

val_dataset = (
    val_dataset.unbatch()
    .shuffle(1000, seed=RANDOM_STATE, reshuffle_each_iteration=False)
    .batch(BATCH_SIZE)
)

test_dataset = val_dataset.take(test_batches)
val_dataset = val_dataset.skip(test_batches)

print(f"Val Batches: {val_batches - test_batches}")
print(f"Test Batches: {test_batches}")

In [ ]:
# Autotuning
autotune = tf.data.AUTOTUNE
train_dataset = train_dataset.prefetch(buffer_size=autotune)
val_dataset = val_dataset.prefetch(buffer_size=autotune)
test_dataset = test_dataset.prefetch(buffer_size=autotune)

In [ ]:
# Plot training-validation accuracy and loss (optionally marking where fine-tuning starts)
def plot_history(acc, val_acc, loss, val_loss, ft_start=None, save_path=None):
    plt.figure(figsize=(8, 8))
    plt.subplot(2, 1, 1)
    plt.title("Training and Validation Accuracy")
    plt.plot(acc, label="Training Accuracy")
    plt.plot(val_acc, label="Validation Accuracy")
    if ft_start is not None:
        plt.plot([ft_start, ft_start], plt.ylim(), label="Start Fine-Tuning")

    plt.ylabel("Accuracy")
    plt.legend(loc="upper left")
    plt.subplot(2, 1, 2)
    plt.title("Training and Validation Loss")
    plt.plot(loss, label="Training Loss")
    plt.plot(val_loss, label="Validation Loss")
    if ft_start is not None:
        plt.plot([ft_start, ft_start], plt.ylim(), label="Start Fine-Tuning")

    plt.xlabel("Epoch")
    plt.ylabel("Binary Cross Entropy")
    plt.legend(loc="upper left")
    if save_path:
        plt.savefig(save_path, bbox_inches="tight")

    plt.show()

In [ ]:
# Base model from the pre-trained model for RGB images
img_shape = IMG_SIZE + (3,)  # 3 channels of color (RGB)
base_model = MobileNetV2(input_shape=img_shape, include_top=False, weights="imagenet")

In [ ]:
# Feature extraction in transfer learning example
image_batch, label_batch = next(iter(train_dataset))
feature_batch = base_model(image_batch)
print(feature_batch.shape)

In [ ]:
# Set base model to not trainable
base_model.trainable = False

In [ ]:
# Model for predictions
# Data augmentation (only active during training) to reduce overfitting
data_augmentation = tf.keras.Sequential(
    [
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.1),
        layers.RandomZoom(0.1),
    ]
)

input = Input(shape=img_shape)
journey = data_augmentation(input)

# Same as mobilenet_v2.preprocess_input: [0, 255] -> [-1, 1]
journey = layers.Rescaling(1 / 127.5, offset=-1)(journey)
journey = base_model(journey, training=False)
journey = layers.GlobalAveragePooling2D()(journey)
journey = layers.Dropout(0.2)(journey)

output = layers.Dense(1, activation="sigmoid")(journey)
curr_model = Model(input, output)

In [ ]:
# Summary curr_model
curr_model.summary()

In [ ]:
# Trainable variables
len(curr_model.trainable_variables)

In [ ]:
# Compile curr_model
curr_model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

In [ ]:
# Initial loss and accuracy
curr_model.evaluate(val_dataset)

In [ ]:
# Callbacks: save the best model (lowest val_loss) across both training phases
# and stop early when val_loss stops improving
checkpoint = callbacks.ModelCheckpoint(
    BEST_MODEL_PATH, monitor="val_loss", save_best_only=True, verbose=1
)
early_stop = callbacks.EarlyStopping(
    monitor="val_loss", patience=5, restore_best_weights=True
)

In [ ]:
# Train the curr_model
init_epoch = 10
curr_history = curr_model.fit(
    train_dataset,
    epochs=init_epoch,
    validation_data=val_dataset,
    verbose=2,
    callbacks=[checkpoint, early_stop],
)

In [ ]:
# Training-Validation result of curr_model
acc = curr_history.history["accuracy"]
loss = curr_history.history["loss"]
val_acc = curr_history.history["val_accuracy"]
val_loss = curr_history.history["val_loss"]
plot_history(
    acc, val_acc, loss, val_loss, save_path=f"{RESULT_DIR}/history_initial.png"
)

In [ ]:
# Set base model to trainable
base_model.trainable = True

In [ ]:
# curr_model summary
curr_model.summary()

In [ ]:
# Fine tuning the base_model
print(f"Number of layers in base model: {len(base_model.layers)}")

In [ ]:
# Freezing the first determined amount of layers
start_ft = 100
for layer in base_model.layers[:start_ft]:
    layer.trainable = False

In [ ]:
# Compile the fine tuning part of curr_model
# Low learning rate so the pre-trained weights are only nudged, not overwritten
curr_model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-6),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

In [ ]:
# Trainable variables from the new compiling
len(curr_model.trainable_variables)

In [ ]:
# Train the fine-tuning curr_model
ft_epoch = 10
total_epoch = len(curr_history.epoch) + ft_epoch
curr_history_ft = curr_model.fit(
    train_dataset,
    epochs=total_epoch,
    initial_epoch=len(curr_history.epoch),
    validation_data=val_dataset,
    verbose=2,
    callbacks=[checkpoint, early_stop],
)

In [ ]:
# Comparison of Training-Validation result of curr_model before and after fine-tuning
acc += curr_history_ft.history["accuracy"]
loss += curr_history_ft.history["loss"]
val_acc += curr_history_ft.history["val_accuracy"]
val_loss += curr_history_ft.history["val_loss"]
plot_history(
    acc,
    val_acc,
    loss,
    val_loss,
    ft_start=len(curr_history.epoch) - 1,
    save_path=f"{RESULT_DIR}/history_fine_tuning.png",
)

In [ ]:
# Load the best model saved by the checkpoint
best_model = tf.keras.models.load_model(BEST_MODEL_PATH)

In [ ]:
# Evaluate best model on the test dataset
best_model.evaluate(test_dataset)

In [ ]:
# Retrieve a batch of images from the test dataset
image_batch, label_batch = test_dataset.as_numpy_iterator().next()
predictions = best_model.predict_on_batch(image_batch).flatten()
print("Real Predictions:\n", predictions)

predictions = tf.where(predictions < 0.5, 0, 1)
print("Predictions:\n", predictions.numpy())
print("Real:\n", label_batch)

plt.figure(figsize=(8, 8))
for i in range(16):
    ax = plt.subplot(4, 4, i + 1)
    plt.imshow(image_batch[i].astype("uint8"))
    plt.title(
        f"P: {class_names[predictions[i]]}, R: {class_names[label_batch[i]]}",
        fontsize=10,
    )
    plt.axis("off")

plt.savefig(f"{RESULT_DIR}/test_predictions.png", bbox_inches="tight")
plt.show()

### Inference ###
Classify any image file with the saved `best_model.keras`. Rescaling is built into the model, so it only needs raw RGB pixels resized to `IMG_SIZE`.

In [ ]:
# Predict the class of a single image file
def predict_image(path, model=best_model):
    img = tf.keras.utils.load_img(path, target_size=IMG_SIZE)  # load + resize
    batch = tf.expand_dims(tf.keras.utils.img_to_array(img), 0)  # (1, H, W, 3)
    prob = float(model.predict(batch, verbose=0)[0][0])  # P(class_names[1])
    label = class_names[int(prob >= 0.5)]
    confidence = prob if prob >= 0.5 else 1 - prob
    return img, label, confidence


img, label, confidence = predict_image("../data/val/bees/abeja.jpg")
plt.imshow(img)
plt.title(f"{label} ({confidence:.1%})")
plt.axis("off")
plt.savefig(f"{RESULT_DIR}/inference_example.png", bbox_inches="tight")
plt.show()

In [ ]:
# Clear Session
tf.keras.backend.clear_session()